# 03. Scheduler、KV Memory Pool 与执行路径

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## Scheduler 关心的不是“一个请求”，而是“资源可行的下一批请求”

每轮循环里 scheduler 都要在这些约束之间取平衡：

- waiting queue 中哪些请求可以进入 prefill。
- running batch 中哪些请求继续 decode。
- KV cache 还有多少空位，可以驱逐多少，哪些节点被锁住。
- grammar 是否已编译完成。
- speculative decoding 是否需要 draft/verify。
- overlap schedule 是否允许 CPU 调度和 GPU 执行重叠。

这就是 SGLang 的吞吐核心：连续 batching、prefix cache、paged KV、chunked prefill 都在 scheduler 汇合。


In [ ]:
for path in [
    "python/sglang/srt/managers/scheduler.py",
    "python/sglang/srt/managers/schedule_batch.py",
    "python/sglang/srt/mem_cache/memory_pool.py",
    "python/sglang/srt/model_executor/forward_batch_info.py",
    "python/sglang/srt/model_executor/model_runner.py",
]:
    print("\n==", path)
    for row in find_defs(path, names={"Scheduler", "Req", "ScheduleBatch", "ReqToTokenPool", "ForwardBatch", "ModelRunner"}):
        print(row)


## `Req`、`ScheduleBatch`、`ForwardBatch`

三个对象对应三个抽象层：

- `Req`：单个请求的 CPU 状态，包含 input ids、output ids、sampling params、prefix hit、grammar、finish reason。
- `ScheduleBatch`：scheduler 选出的一个 batch，仍以 Python/CPU 状态为主，但已经决定了 extend/decode、KV 分配等。
- `ForwardBatch`：model runner 消费的 tensor 化执行元数据，attention backend 依赖它知道 seq_lens、cache loc、forward mode。


### `Req.__init__`：单个请求的“账本”

```python
# python/sglang/srt/managers/schedule_batch.py:666-745
 666: class Req(ReqDllmMixin):
 667:     """The input and output status of a request."""
 668: 
 669:     def __init__(
 670:         self,
 671:         rid: str,
 672:         origin_input_text: str,
 673:         origin_input_ids: array[int],
 674:         sampling_params: SamplingParams,
 675:         return_logprob: bool = False,
 676:         top_logprobs_num: int = 0,
 677:         dllm_config: Optional[DllmConfig] = None,
 678:         token_ids_logprob: List[int] = None,
 679:         stream: bool = False,
 680:         origin_input_ids_unpadded: Optional[array[int]] = None,
 681:         lora_id: Optional[str] = None,
 682:         input_embeds: Optional[List[List[float]]] = None,
 683:         positional_embed_overrides: Optional[PositionalEmbeds] = None,
 684:         token_type_ids: List[int] = None,
 685:         session: Optional[Session] = None,
 686:         custom_logit_processor: Optional[str] = None,
 687:         require_reasoning: bool = False,
 688:         return_hidden_states: bool = False,
 689:         return_routed_experts: bool = False,
 690:         routed_experts_start_len: int = 0,
 691:         return_indexer_topk: bool = False,
 692:         eos_token_ids: Optional[Set[int]] = None,
 693:         bootstrap_host: Optional[str] = None,
 694:         bootstrap_port: Optional[int] = None,
 695:         bootstrap_room: Optional[int] = None,
 696:         disagg_mode: Optional[DisaggregationMode] = None,
 697:         routed_dp_rank: Optional[int] = None,
 698:         disagg_prefill_dp_rank: Optional[int] = None,
 699:         vocab_size: Optional[int] = None,
 700:         priority: Optional[int] = None,
 701:         metrics_collector: Optional[SchedulerMetricsCollector] = None,
 702:         extra_key: Optional[str] = None,
 703:         routing_key: Optional[str] = None,
 704:         dimensions: Optional[int] = None,
 705:         http_worker_ipc: Optional[str] = None,
 706:         time_stats: Optional[
 707:             Union[APIServerReqTimeStats, DPControllerReqTimeStats]
 708:         ] = None,
 709:         return_pooled_hidden_states: bool = False,
 710:         multi_item_delimiter_indices: Optional[List[int]] = None,
 711:     ):
 712:         # Input and output info
 713:         self.rid = rid
      # 注：请求全局 id，是 scheduler 输出回到 TokenizerManager `rid_to_state` 的路由键。
 714:         self.origin_input_ids = origin_input_ids
      # 注：原始 prompt token ids；prefix cache、长度校验和 prefill 都以它为基础。
 715:         self.origin_input_ids_unpadded = (
      # 注：多模态 padding 前的原始 token ids，用于需要还原用户输入长度的统计或返回。
 716:             origin_input_ids_unpadded
 717:             if origin_input_ids_unpadded
 718:             else self.origin_input_ids
 719:         )  # Before image padding
 720:         # Each decode stage's output ids. Append-only by contract:
 721:         # _refresh_fill_ids infers how many output tokens are already in
 722:         # full_untruncated_fill_ids from lengths alone, so in-place rewrites
 723:         # that preserve length would silently corrupt fill_ids.
 724:         self.output_ids = array("q")
      # 注：生成阶段 append-only 的输出 token；scheduler 根据长度差推断新增 token，不能原地改写。
 725:         # Full untruncated sequence: origin + output (+ DLLM mask block).
 726:         # Kept in sync by _refresh_fill_ids; admission only updates fill_len,
 727:         # never mutates this array's length.
 728:         self.full_untruncated_fill_ids = array("q")
      # 注：prompt+output 的完整序列镜像，chunked prefill/DLLM 等路径用它恢复 fill 状态。
 729:         self.fill_len: int = 0
      # 注：当前已经进入 fill/prefill 处理的 token 长度，admission 不直接截断完整序列。
 730: 
 731:         self.session = session
 732:         self.input_embeds = input_embeds
 733:         self.positional_embed_overrides = positional_embed_overrides
 734:         self.multi_item_delimiter_indices = multi_item_delimiter_indices
 735: 
 736:         # For req-level memory management
 737:         self.kv_committed_len = 0
      # 注：已提交、语义上可保留的 KV 长度；finished/cache 路径只处理这部分 KV。
 738:         self.kv_allocated_len = 0
      # 注：已经向 KV pool 申请的长度，可能大于 committed 长度，用于处理预分配和回收。
 739:         self.kv_committed_freed = False
      # 注：记录 committed KV 是否已释放，防止 abort/finish 多路径重复 free。
 740:         self.kv_overallocated_freed = False
      # 注：记录 overallocated KV 是否已释放，避免预分配槽位泄漏或重复释放。
 741: 
 742:         # for corss-endoder model
 743:         self.token_type_ids = token_type_ids
 744: 
 745:         # The length of KV that have been removed in swa cache.
```

**读这段时抓住：**

- `origin_input_ids` 是 prompt，`output_ids` 是 append-only 的生成结果；许多后续长度计算假设 output 只追加不重写。
- `kv_committed_len` / `kv_allocated_len` 区分已经提交给请求语义的 KV 和临时/预分配 KV。
- `extra_key`、`lora_id`、`routing_key` 等字段决定调度、cache 隔离和分布式路由，不只是 API 元数据。


### `ScheduleBatch`：scheduler 侧 batch 同时拿着资源和请求

```python
# python/sglang/srt/managers/schedule_batch.py:1671-1738
1671: class ScheduleBatch(ScheduleBatchDisaggregationDecodeMixin):
1672:     """Store all information of a batch on the scheduler."""
1673: 
1674:     # === Core: request list (ForwardBatch derives lora_ids / rids / grammars / positions from it) ===
1675:     reqs: List[Req]
      # 注：ScheduleBatch 中的请求列表；ForwardBatch 会从它派生 LoRA、grammar、position 等 per-request 元数据。
1676: 
1677:     # === Global config and shared resources (engine-lifetime; identical across batches) ===
1678:     # Memory pool and cache
1679:     req_to_token_pool: ReqToTokenPool = None
      # 注：请求逻辑 token 位置到 KV slot 的映射池，batch 执行和 prefix cache 都会读取它。
1680:     token_to_kv_pool_allocator: BaseTokenToKVPoolAllocator = None
      # 注：物理 KV slot allocator，负责为 extend/decode 分配或释放 KV 存储。
1681:     tree_cache: BasePrefixCache = None
      # 注：batch 共享的 prefix cache 引用，finished/unfinished request 会通过它更新 radix cache。
1682: 
1683:     # Batch configs
1684:     model_config: ModelConfig = None
1685:     enable_overlap: bool = False
1686: 
1687:     # Device
1688:     device: str = "cuda"
1689: 
1690:     # HiSparse (engine-level coordinator ref, same across batches)
1691:     hisparse_coordinator: Optional[HiSparseCoordinator] = None
1692: 
1693:     # === Batch-variant scheduler state (per-batch; not read by ForwardBatch) ===
1694:     # Tell whether the current running batch is full so that we can skip
1695:     # the check of whether to prefill new requests.
1696:     # This is an optimization to reduce the overhead of the prefill check.
1697:     batch_is_full: bool = False
      # 注：标记 running batch 是否已满；为满时可跳过新 prefill 检查，减少 scheduler 热路径开销。
1698: 
1699:     # For chunked prefill in PP
1700:     chunked_req: Optional[Req] = None
      # 注：当前被切块 prefill 的请求，下一轮会继续从它的后续 prompt token 开始。
1701:     chunked_req_next_prompt_token: Optional[int] = None
1702:     contains_last_prefill_chunk: bool = True
1703: 
1704:     # For DP attention
1705:     inner_idle_batch: Optional[ScheduleBatch] = None
1706:     # Decode requests carried alongside a chunked-prefill batch
1707:     decoding_reqs: List[Req] = None
      # 注：与 chunked-prefill batch 同时携带的 decode 请求，用于混合 prefill/decode 调度。
1708: 
1709:     # For split prefill
1710:     split_index: int = 0
1711:     split_prefill_finished: bool = False
1712:     split_forward_count: int = 1
1713:     split_forward_batch: ForwardBatch = None
1714: 
1715:     # CPU mirror of req_pool_indices; schedule-path only (used in overlap_utils,
1716:     # not read by ForwardBatch), stale in spec draft window
1717:     req_pool_indices_cpu: torch.Tensor = None  # shape: [b], int64
      # 注：req_pool_indices 的 CPU 镜像，overlap utils 使用；spec draft window 中可能滞后于 GPU tensor。
1718: 
1719:     # Forward-pass metrics
1720:     fpm_start_time: float = 0.0
1721: 
1722:     # hicache pointer for synchronizing data loading from CPU to GPU
1723:     hicache_consumer_index: int = -1
      # 注：HiCache load-back 与 batch 消费同步用的指针，避免 GPU 在 KV 搬回前读取。
1724: 
1725:     # Metrics
1726:     dp_cooperation_info: Optional[DPCooperationInfo] = None
1727:     prefill_stats: Optional[PrefillStats] = None
1728:     forward_iter: Optional[int] = None
1729: 
1730:     # === GPU tensors crossing to ForwardBatch (clone targets for stream isolation) ===
1731:     # Batched arguments to model runner
1732:     input_ids: torch.Tensor = None  # shape: [b], int64
      # 注：传给 model runner 的本轮输入 token tensor，是 ScheduleBatch 到 ForwardBatch 的核心数据。
1733:     # Staging consumed by resolve_forward_inputs (prefill H2D / mixed gather).
1734:     prefill_input_ids_cpu: Optional[torch.Tensor] = None
      # 注：prefill H2D staging 区，resolve_forward_inputs 会消费它构造最终 GPU input_ids。
1735:     mix_running_indices: Optional[torch.Tensor] = None
      # 注：混合 prefill/decode 时用于 gather running token 的索引，resolve_forward_inputs 会读取它。
1736:     input_embeds: torch.Tensor = None  # shape: [b, hidden_size], float32
1737: 
1738:     # Token replacement embeddings and absolute positions (optional).
```

**读这段时抓住：**

- `reqs` 是高层请求列表；`req_to_token_pool`、`token_to_kv_pool_allocator`、`tree_cache` 是共享资源引用。
- batch-variant 字段记录 chunked prefill、split prefill、HiCache consumer index、metrics 等调度状态。
- `input_ids`、`req_pool_indices` 等 tensor 字段是跨到 `ForwardBatch` 的桥，不是简单 Python list。


### `ForwardBatch`：attention backend 真正读取的执行快照

```python
# python/sglang/srt/model_executor/forward_batch_info.py:322-388
 322: class ForwardBatch(ForwardBatchDeepSeekMHAMixin):
 323:     """Store all inputs of a forward pass."""
 324: 
 325:     # === Required core inputs (no default; input_ids / req_pool_indices / seq_lens / out_cache_loc are borrowed from ScheduleBatch) ===
 326:     # The forward mode
 327:     forward_mode: ForwardMode
      # 注：ForwardBatch 的执行模式，attention backend 依赖它区分 prefill/decode/verify/extend。
 328:     # The batch size
 329:     batch_size: int
      # 注：本次 forward 的请求数或执行行数，kernel launch 和 sampling batch 都依赖它。
 330:     # The input ids
 331:     input_ids: torch.Tensor
      # 注：传给 model runner 的本轮输入 token tensor，是 ScheduleBatch 到 ForwardBatch 的核心数据。
 332:     # The indices of requests in the req_to_token_pool
 333:     req_pool_indices: torch.Tensor
      # 注：ForwardBatch 中每个请求在 req_to_token_pool 的行索引，attention backend 用它定位历史 KV。
 334:     # The sequence length
 335:     seq_lens: torch.Tensor
      # 注：每个请求当前序列长度，paged attention 和 sampling 都依赖它计算有效上下文。
 336:     # The indices of output tokens in the token_to_kv_pool
 337:     out_cache_loc: torch.Tensor
      # 注：本轮输出 token 要写入的 KV slot 位置，是 attention 写 KV 的目标地址。
 338:     # The sum of all sequence lengths
 339:     seq_lens_sum: int
      # 注：batch 内总 token 长度，部分 attention backend 用它规划 prefill kernel。
 340: 
 341:     # === Borrowed from ScheduleBatch: GPU tensors (cross-stream; clone targets for stream isolation) ===
 342:     # FIXME(lsyin): these are currently aliased by reference from ScheduleBatch. Once
 343:     # they are cloned/relayed into FB-owned copies at the boundary, move them out of
 344:     # "Borrowed" into a dedicated "Forward-resolved snapshot" group.
 345:     # The original sequence length without being chunked. Qwen-1M related.
 346:     orig_seq_lens: Optional[torch.Tensor] = None
      # 注：chunked prefill 前的原始长度，长上下文模型和日志统计需要保留它。
 347: 
 348:     # DSV4-NPU only: per-pool slot bundle from DSV4NPUTokenToKVPoolAllocator,
 349:     # consumed by the Ascend backend for PA_ND block tables. None elsewhere.
 350:     out_cache_loc_dsv4: Optional[DSV4OutCacheLoc] = None
 351:     # The indices to track mamba state with
 352:     mamba_track_indices: Optional[torch.Tensor] = None  # shape: [b], int64
 353:     # The mask to track mamba state if needed
 354:     mamba_track_mask: Optional[torch.Tensor] = None  # shape: [b], bool
 355:     # The seqlens to track mamba state if masked, prefill only.
 356:     mamba_track_seqlens: Optional[torch.Tensor] = None  # shape: [b], int64
 357:     # Deferred mamba init ops: COW pairs and clear indices (performed on forward stream)
 358:     mamba_cow_src_indices: Optional[torch.Tensor] = None
 359:     mamba_cow_dst_indices: Optional[torch.Tensor] = None
 360:     mamba_clear_indices: Optional[torch.Tensor] = None
 361: 
 362:     # For input embeddings
 363:     input_embeds: Optional[torch.Tensor] = None
 364:     # For token embedding overrides (sparse replacement at specific positions)
 365:     replace_embeds: Optional[torch.Tensor] = None
 366:     replace_positions: Optional[torch.Tensor] = None
 367: 
 368:     # For cross-encoder model
 369:     token_type_ids: Optional[torch.Tensor] = None
 370: 
 371:     # Encoder-decoder device tensors
 372:     encoder_lens: Optional[torch.Tensor] = None
 373:     encoder_out_cache_loc: Optional[torch.Tensor] = None
 374: 
 375:     # === Borrowed from ScheduleBatch: config / flags (by-value) ===
 376:     # For logprob
 377:     return_logprob: bool = False
      # 注：是否需要返回 logprob；为 True 时 model runner/sampling 会保留额外概率信息。
 378:     # Whether this batch is prefill-only (no token generation needed)
 379:     is_prefill_only: bool = False
      # 注：该 batch 只做 prefill 不采样生成 token，常见于 embedding 或预填充分离路径。
 380:     spec_algorithm: SpeculativeAlgorithm = None
      # 注：当前 batch 使用的 speculative algorithm，决定 verify/draft worker 和 accept 逻辑。
 381:     # For matryoshka embeddings
 382:     dimensions: Optional[list[int]] = None
 383:     # Whether to return pooled hidden states (pre-head transformer output)
 384:     return_pooled_hidden_states: bool = False
 385: 
 386:     # For DP attention
 387:     is_extend_in_batch: bool = False
      # 注：batch 中存在 extend/prefill 请求，attention backend 需要按 extend 而非纯 decode 处理。
 388:     # Mirrors ScheduleBatch.all_extend_in_batch; kept for downstream forks.
```

**读这段时抓住：**

- `forward_mode`、`seq_lens`、`out_cache_loc` 是 attention backend 决定 prefill/decode/verify 行为的核心。
- 许多字段从 `ScheduleBatch` 借用而非复制，说明 stream isolation 和生命周期管理非常敏感。
- speculative、DP attention、Mamba、encoder-decoder 都在这里挂执行元数据，所以新增执行路径往往会扩展 ForwardBatch。


In [ ]:
for name in ["Req", "ScheduleBatch"]:
    rows = find_defs("python/sglang/srt/managers/schedule_batch.py", {name})
    print(name, rows)
for name in ["ForwardBatch", "ForwardMode"]:
    rows = find_defs("python/sglang/srt/model_executor/forward_batch_info.py", {name})
    print(name, rows)


In [ ]:
show_lines("python/sglang/srt/managers/schedule_batch.py", 1, 35)
show_lines("python/sglang/srt/managers/schedule_batch.py", 35, 75)


## KV memory pool：为什么需要两层映射

LLM serving 中 KV cache 是巨大的 `[layers, tokens, heads, dim]` 存储。SGLang 会把请求逻辑位置映射到物理 KV slot：

- `ReqToTokenPool`：请求维度的 token position -> KV slot index。
- `TokenToKVPool` / allocator：KV slot index -> 每层 K/V tensor 中的实际位置。

这样 decode 时新增 token 只需追加 slot，prefix cache 命中时可以直接复用已有 slot indices。


In [ ]:
for path in [
    "python/sglang/srt/mem_cache/memory_pool.py",
    "python/sglang/srt/mem_cache/allocator/base.py",
    "python/sglang/srt/mem_cache/common.py",
]:
    print("\n==", path)
    for lineno, kind, name in find_defs(path):
        if "Pool" in name or name in {"alloc_for_extend", "alloc_for_decode", "release_kv_cache", "evict_from_tree_cache"}:
            print(f"{lineno:4d} {kind:16s} {name}")


## prefill 与 decode 的调度差异

- **Prefill / extend**：一次处理 prompt 或 chunk，可能写入大量 KV；受 `max_prefill_tokens`、chunked prefill、prefix hit、grammar readiness 影响。
- **Decode**：每个 running request 通常前进一步；关注 `max_running_requests`、KV reserve、sampling、finish。

Speculative decoding 会把 decode 变复杂：一次可能 draft 多个 token，再由 target verify 接受若干 token。


In [ ]:
show_lines("python/sglang/srt/managers/scheduler.py", 2735, 2795)
show_lines("python/sglang/srt/managers/scheduler.py", 3178, 3225)


### `get_new_batch_prefill`：新请求进入 GPU 前的门禁

```python
# python/sglang/srt/managers/scheduler.py:2735-2795
2735:     def get_new_batch_prefill(self) -> Optional[ScheduleBatch]:
2736:         prefill_delayer_single_pass = None
2737:         if self.prefill_delayer:
2738:             # Get max usage across all pools for prefill delay decision
2739:             max_pool_usage = (
2740:                 self.pool_stats_observer.get_pool_stats().get_max_pool_usage()
2741:             )
2742:             prefill_delayer_single_pass = PrefillDelayerSinglePassExecutor(
2743:                 self.prefill_delayer, token_usage=max_pool_usage
2744:             )
2745: 
2746:         ret = self._get_new_batch_prefill_raw(
2747:             prefill_delayer_single_pass=prefill_delayer_single_pass
2748:         )
2749: 
2750:         if self.prefill_delayer:
2751:             prefill_delayer_single_pass.finalize(actual_prefill=ret is not None)
2752: 
2753:         return ret
2754: 
2755:     def _get_new_batch_prefill_raw(
2756:         self, prefill_delayer_single_pass: Optional[PrefillDelayerSinglePassExecutor]
2757:     ) -> Optional[ScheduleBatch]:
2758:         # Check if the grammar is ready in the grammar queue
2759:         if self.grammar_manager.has_waiting_grammars():
2760:             ready_grammar_requests = self.grammar_manager.get_ready_grammar_requests()
2761:             for req in ready_grammar_requests:
2762:                 self._add_request_to_queue(req)
2763: 
2764:         if self.enable_hierarchical_cache:
2765:             self.tree_cache.check_hicache_events()
2766: 
2767:         if self.enable_priority_preemption or self.is_hybrid_swa:
2768:             # Reset batch_is_full to try preemption with a prefill adder.
2769:             self.running_batch.batch_is_full = False
2770: 
2771:         if (
2772:             self.running_batch.batch_is_full or len(self.waiting_queue) == 0
2773:         ) and self.chunked_req is None:
2774:             return None
2775: 
2776:         running_bs = len(self.running_batch.reqs)
2777:         if self._should_delay_dflash_prefill_for_batching(running_bs):
2778:             return None
2779: 
2780:         # Ignore the check if self.chunked_req is not None.
2781:         # In the non-PP case, when self.chunked_req is not None, num_allocatable_reqs should always be greater than 0,
2782:         # as the space for the chunked requests has just been released.
2783:         # In PP case, chunked requests (or dllm requests) can start in one microbatch and end in another microbatch, so the max_running_requests per microbatch should not be strict.
2784:         # Instead, we should always allow chunked requests to be added, otherwise, there will be a memory leak.
2785:         if (
2786:             self.get_num_allocatable_reqs(running_bs) <= 0
2787:             and self.chunked_req is None
2788:             and not self.enable_priority_preemption
2789:         ):
2790:             self.running_batch.batch_is_full = True
2791:             return None
2792: 
2793:         # Get priority queue
2794:         self.policy.calc_priority(self.waiting_queue, self.running_batch)
2795: 
```

**读这段时抓住：**

- 它不是简单 pop waiting queue，而要同时检查 grammar、memory、chunked prefill、disaggregation、priority 等条件。
- 这里形成的是 prefill/extend batch；decode 的 running batch 在另一条路径继续推进。
- 如果你新增调度策略，通常要看它在这段之前还是之后改变 waiting queue。


### `run_batch`：scheduler 和 model worker 的窄腰

```python
# python/sglang/srt/managers/scheduler.py:3178-3238
3178:     def run_batch(
3179:         self,
3180:         batch: ScheduleBatch,
3181:         pp_proxy_tensors: Optional[PPProxyTensors] = None,
3182:     ) -> Union[GenerationBatchResult, EmbeddingBatchResult]:
3183:         """Run a batch."""
3184:         self.forward_ct += 1
3185:         batch.forward_iter = self.forward_ct
      # 注：把 scheduler 全局 forward 计数写入 batch，metrics/profile 可用它关联一次 GPU 执行。
3186: 
3187:         if self.scripted_scheduler_hook is not None:
      # 注：存在脚本化 scheduler hook 时，每个 batch 执行前都给实验代码一个观察/干预点。
3188:             self.scripted_scheduler_hook.on_run_batch(batch)
      # 注：关键调用：`self.scripted_scheduler_hook.on_run_batch` - 测试/脚本 hook 可在每次 batch 执行前观察或修改 batch，用于调度实验和可控复现。
3189: 
3190:         # Whether to run the profiler
3191:         self.profiler_manager._profile_batch_predicate(batch)
      # 注：关键调用：`self.profiler_manager._profile_batch_predicate` - 按当前 batch 决定是否启动 profiler，避免对所有请求都付 profiling 开销。
3192:         if self.forward_sleep_time is not None:
      # 注：调试用人为 sleep，可放大并发/overlap 时序问题，生产路径不应开启。
3193:             logger.info(f"Scheduler.run_batch sleep {self.forward_sleep_time}s")
3194:             time.sleep(self.forward_sleep_time)
3195: 
3196:         # Place holder handling for pd-disagg decode event loop
3197:         if batch.forward_mode.is_prebuilt():
      # 注：prebuilt batch 来自 disaggregation decode 等外部构造路径，scheduler 不再按普通 batch 重新准备输入。
3198:             return self._run_batch_prebuilt(batch)
      # 注：关键调用：`self._run_batch_prebuilt` - PD disaggregation decode 可以直接使用预构建 batch，绕过普通 scheduler 现场组装路径。
3199: 
3200:         # Run forward
3201:         if self.is_generation:
      # 注：生成模型走 token generation forward；embedding/score 等非生成任务会走 run_batch 的其他分支。
3202:             if self.enable_overlap:
      # 注：overlap 开启时 CPU 调度和 GPU forward 分离，需要 future_map、stream wait 和状态隔离配合。
3203:                 # Self-gates on batch.spec_info.future_indices; non-spec_v2
3204:                 # no-ops (ForwardBatch.init_new lazily computes the sum).
3205:                 self.future_map.resolve_seq_lens_cpu(batch)
      # 注：关键调用：`self.future_map.resolve_seq_lens_cpu` - overlap 模式下先解析 speculative future indices 对 seq_lens 的影响，保证 CPU 调度看到正确长度。
3206: 
3207:                 with self.forward_stream_ctx:
3208:                     self.forward_stream.wait_stream(self.schedule_stream)
3209:                     # resolve consumes SB staging (prefill_input_ids_cpu /
3210:                     # mix_running_indices). Run OUTSIDE isolation so the
3211:                     # snapshot captures the post-consume state — restoring
3212:                     # post-forward must not un-consume staging.
3213:                     resolve_forward_inputs(batch, self.future_map)
      # 注：关键调用：`resolve_forward_inputs` - 把 ScheduleBatch 中的 CPU staging 输入解析成 model worker 可消费的 forward 输入。
3214: 
3215:                     with self._forward_isolation(batch, overlap=True):
3216:                         future_indices = batch.req_pool_indices
      # 注：overlap/speculative 发布用的 req_pool_indices 快照，worker 执行期间 scheduler 依赖它更新 future_map。
3217: 
3218:                         # Spec_v2 fires on_publish mid-worker (between verify and
3219:                         # draft_extend) so schedule prep can overlap with draft_extend.
3220:                         # Non-spec has no later work — scheduler publishes after return.
3221:                         fwd_kwargs = (
      # 注：speculative overlap 时传给 model worker 的 publish hook；非 spec batch 不需要额外 kwargs。
3222:                             {
3223:                                 "on_publish": partial(
3224:                                     self.future_map.publish, future_indices
      # 注：关键调用：`self.future_map.publish` - overlap/speculative 路径把可见 seq_lens 发布给调度线程，使下一轮 CPU 准备可提前进行。
3225:                                 )
3226:                             }
3227:                             if not batch.spec_algorithm.is_none()
      # 注：speculative batch 需要在 target verify 与 draft extend 之间提前 publish，给 scheduler overlap 留窗口。
3228:                             else {}
3229:                         )
3230: 
3231:                         # FIXME: pp is not compatible with overlap
3232:                         batch_result = self.model_worker.forward_batch_generation(
      # 注：model worker 返回的生成结果，后续 scheduler 会据此更新 token、finish reason、KV cache 和回包。
3233:                             batch, **fwd_kwargs
3234:                         )
3235:                         if batch.spec_algorithm.is_none():
      # 注：非 speculative batch 没有 draft 后续工作，model worker 返回后再发布 seq_lens 即可。
3236:                             self.future_map.publish(future_indices, batch.seq_lens + 1)
      # 注：关键调用：`self.future_map.publish` - overlap/speculative 路径把可见 seq_lens 发布给调度线程，使下一轮 CPU 准备可提前进行。
3237:                         # Park any refs the worker wants kept alive 2 iters
3238:                         # (cross-stream tensor lifetime; pinned in the same
```

**读这段时抓住：**

- `run_batch` 接收 `ScheduleBatch`，根据模式走普通 forward、speculative forward、embedding/score 等路径。
- 它是 GPU 执行的调用点，但仍在 scheduler 进程里负责前后状态 bookkeeping。
- 很多性能 Feature 会在这里附近插入计时、profile、overlap 或特殊 forward mode。


## ModelRunner：从 batch 到模型 forward

`ModelRunner` 是权重、attention backend、CUDA graph runner、forward context 的聚合点。新增模型时通常在 `srt/models` 实现模型结构；新增执行优化时通常会碰 `model_executor` 或 attention backend。


In [ ]:
for name in ["ModelRunner", "forward", "forward_decode", "forward_extend"]:
    rows = find_defs("python/sglang/srt/model_executor/model_runner.py", {name})
    print(name, rows[:5])


## 小练习：用源码检查 Scheduler 主循环有哪些旁路 Feature

这个 cell 抓取 `Scheduler.__init__` 附近常见子系统名。它不是完整解析器，但能帮助你建立“Feature 都挂在哪里”的直觉。


In [ ]:
scheduler_src = read_rel("python/sglang/srt/managers/scheduler.py")
keywords = [
    "grammar", "spec", "lora", "disaggregation", "hicache",
    "overlap", "cuda_graph", "metrics", "profile", "cache_controller",
]
for kw in keywords:
    hits = [i+1 for i, line in enumerate(scheduler_src.splitlines()) if kw in line.lower()]
    print(f"{kw:18s}", hits[:10], "..." if len(hits) > 10 else "")


## 贡献者注意点

- Scheduler 里的 bug 往往是状态生命周期 bug：请求 abort、streaming、chunked prefill、grammar pending、spec verify 都可能改变同一个 batch。
- KV 分配失败不一定是 OOM，也可能是 prefix cache 锁住了太多节点或 page reserve 估计不对。
- 改采样参数时要同时检查 tokenizer manager 的输入校验、scheduler 的 batch 合并、sampling batch info 的 tensor 化。
